In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
# Load local HTML file and extract URLs
base_url = "https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/"

with open('chtml_tme16_source_list.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')

# Find all <p> tags containing "Fn and Ft:"
urls_to_fetch = []
for p_tag in soup.find_all('p'):
    text = p_tag.get_text()
    if 'Fn and Ft:' in text:
        link = p_tag.find('a', href=True)
        if link:
            href = link['href']
            full_url = base_url + href
            urls_to_fetch.append(full_url)

# print(f"Found {len(urls_to_fetch)} URLs")
# for url in urls_to_fetch[:5]:
#     print(url)
# print("...")
# print(f"Last URL: {urls_to_fetch[-1]}")

In [3]:
urls_to_fetch

['https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/ANOPRAIM_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/ANOPRAIM_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/LECPROV1_MLBLR18.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/LECPROV2_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/LECPROV3_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/LECPROV4_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/SCOTA3B1_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/SCOTA3B2_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/SCOTA3B3_TEXT.html',
 'https://web.archive.org/web/20240806170534/https://chmtl.indiana.edu/tme/16th/SCOTA3B4

In [4]:
import os
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Create output folder if it doesn't exist
output_folder = 'english_sources'
os.makedirs(output_folder, exist_ok=True)

# Set up a session with retry logic
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,  # Wait 2, 4, 8 seconds between retries
    status_forcelist=[429, 500, 502, 503, 504]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

# Fetch and save each URL
for i, url in enumerate(urls_to_fetch):
    filename = url.split('/')[-1]
    filepath = os.path.join(output_folder, filename)
    
    if os.path.exists(filepath):
        print(f"Skipping {filename} (already exists)")
        continue
    
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[{i+1}/{len(urls_to_fetch)}] Saved: {filename}")
        
        # Longer delay to avoid rate limiting (3 seconds)
        time.sleep(3)
        
    except requests.RequestException as e:
        print(f"Error fetching {filename}: {e}")

print(f"\nDone! Files saved to '{output_folder}/' folder")

Skipping ANOPRAIM_TEXT.html (already exists)
Skipping ANOPRAIM_TEXT.html (already exists)
Skipping LECPROV1_MLBLR18.html (already exists)
Skipping LECPROV2_TEXT.html (already exists)
Skipping LECPROV3_TEXT.html (already exists)
Skipping LECPROV4_TEXT.html (already exists)
Skipping SCOTA3B1_TEXT.html (already exists)
Skipping SCOTA3B2_TEXT.html (already exists)
Skipping SCOTA3B3_TEXT.html (already exists)
Skipping SCOTA3B4_TEXT.html (already exists)
Skipping SCOTANO2_TEXT.html (already exists)
Skipping TUCKE_TEXT.html (already exists)
Skipping TUCKEM_MLBLA103.html (already exists)
Skipping BATHBISS_TEXT.html (already exists)
Skipping BATHBITA_TEXT.html (already exists)
Skipping CORNPAR1_MLBLR18.html (already exists)
Skipping CORNPAR2_MLBLH43.html (already exists)
Skipping CORNPAR3_TEXT.html (already exists)
Skipping CORNPAR4_TEXT.html (already exists)
Skipping CORNPAR5_TEXT.html (already exists)
Skipping DEEPRE_TEXT.html (already exists)
[22/30] Saved: HAWESPP1_TEXT.html
[23/30] Saved: 

In [ ]:
# Fetch TMI (Thesaurus Musicarum Italicarum) texts
import os
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# First, get the index page and extract links
tmi_index_url = "https://tmiweb.science.uu.nl/text/index.html"
tmi_base_url = "https://tmiweb.science.uu.nl/text/"

response = requests.get(tmi_index_url)
soup = BeautifulSoup(response.content, 'html.parser')

# Extract all reading-edition links (excluding the index itself)
tmi_links = [
    tmi_base_url + link['href']
    for link in soup.find_all('a', href=True)
    if link['href'].startswith('reading-edition/') 
    and link['href'] != 'reading-edition/index.html'
]

print(f"Found {len(tmi_links)} TMI text links")
for link in tmi_links[:5]:
    print(f"  {link}")
print("...")

In [ ]:
# Fetch and save TMI HTML files
output_folder = 'tmi_sources'
os.makedirs(output_folder, exist_ok=True)

# Set up session with retry logic
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

# Fetch and save each URL
for i, url in enumerate(tmi_links):
    filename = url.split('/')[-1]
    filepath = os.path.join(output_folder, filename)
    
    if os.path.exists(filepath):
        print(f"Skipping {filename} (already exists)")
        continue
    
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[{i+1}/{len(tmi_links)}] Saved: {filename}")
        
        # Delay to be polite to the server
        time.sleep(1)
        
    except requests.RequestException as e:
        print(f"Error fetching {filename}: {e}")

print(f"\nDone! Files saved to '{output_folder}/' folder")